In [1]:
import copy
import networkx as nx
import os
import pandas as pd
import statsmodels.api as sm
import sys
import warnings

from dowhy import CausalModel
from pathlib import Path

module_path = os.path.abspath(os.path.join("../"))
if module_path not in sys.path:
    sys.path.append(module_path)

from config_loaders.rk_config_loader import RKConfig
from utils.data_io import load_jsonl, make_dir
from utils.vis import save_graph_to_img

warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniforge/base/envs/mllm_diag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROMPT_VERSION = 4
MODEL = "internvl2"
DATASET = "vqav2"

In [3]:
rk_config = RKConfig()
ds_list = rk_config.get_ds_list(MODEL)

In [4]:
# TODO: Correct out path
CE_OUT_DIR = "./causal_effects"
make_dir(CE_OUT_DIR)

[Errno 17] File exists: 'causal_effects'


Load responses_counter
- These files already include the answer for the "complete RK"

In [5]:
# TODO: change this to Path(__file__).parent for script
base_dir = Path("../")
resps_path = rk_config.get_parsed_rk_paths(MODEL, DATASET).format(PROMPT_VERSION)
resps_path = base_dir.joinpath(resps_path)
print(resps_path)
resps = load_jsonl(resps_path)

alt_resps_path = rk_config.get_counterfactual_rk_paths(MODEL, DATASET)
alt_resps_path = base_dir.joinpath(alt_resps_path)
print(alt_resps_path)
alt_resps = load_jsonl(alt_resps_path)

../data/really_know/parsed/internvl2/vqav2/results_v4.jsonl
../data/really_know/counterfactual/internvl2/vqav2/results.jsonl


In [6]:
resps[0].keys()

dict_keys(['question_id', 'response', 'rationales', 'triples', 'triple_objs'])

In [7]:
alt_resps[0].keys()

dict_keys(['question_id', 'size_treatment', 'occluded_concepts', 'path', 'response'])

In [8]:
# Get responses only for the max num of triplets
# (i.e., ignore the rest of the powerset for now)
resps_ids = set([q["question_id"] for q in resps])
alt_resps_ids = set([q["question_id"] for q in alt_resps])

# question_id: response
dict_eval_resp = {
    q_idx: next(r["response"] for r in resps if r["question_id"] == q_idx)
    for q_idx in resps_ids
}

# question_id: response
# this only gets the first response
dict_counter_resp = {
    q_idx: next(r["response"] for r in alt_resps if r["question_id"] == q_idx)
    for q_idx in alt_resps_ids
}

For each question, add a new entry corresponding to the original response.

In [9]:
resps[0].keys()

dict_keys(['question_id', 'response', 'rationales', 'triples', 'triple_objs'])

In [ ]:
# Based on the counterfactual responses we have, pick and add the original ones.
all_resps = copy.deepcopy(alt_resps)
for q_idx in alt_resps_ids:
    og_response = next((r for r in resps if r["question_id"] == q_idx), None)
    if og_response:
        all_resps.append(
            {
                "question_id": og_response["question_id"],
                "size_treatment": 0,
                "occluded_concepts": [],
                "response": og_response["response"],
            }
        )
    else:
        print(f"We have an issue with question {q_idx}.")

Create dataframe out of counterfactual data and responses

In [11]:
df = pd.DataFrame(all_resps)
df.drop(["path"], axis=1, inplace=True)

In [12]:
df.head()

,question_id,size_treatment,occluded_concepts,response
0,337814001,1,[28-table],3
1,337814001,1,[6-wall],3
2,337814001,1,[26-chair],3
3,337814001,1,[2-leg],3
4,337814001,1,[1-wall],3


In [ ]:
# Create mapping between questions and the biggest size of constraints.
dict_max_triplets = {
    q_idx: max(
        [int(q["size_treatment"]) for q in alt_resps if q["question_id"] == q_idx]
    )
    for q_idx in alt_resps_ids
}

# Make dictionary with {question_id: [concepts]}.
# This is done to extract an actual list of concepts, not any other PD objects.
dict_concepts = {
    q_idx: df[
        (df["question_id"] == q_idx)
        & (df["size_treatment"] == dict_max_triplets[q_idx])
    ]["occluded_concepts"].iloc[0]
    for q_idx in dict_max_triplets
}

Add column ```y``` representing the model answer
- If response is numeric, just use that
- If response is text, create lookup for the results of that question

In [14]:
lookup_y = {}

for idx, row in df.iterrows():
    q_idx = row["question_id"]
    if q_idx not in lookup_y:
        lookup_y[q_idx] = {}

    try:
        y = pd.to_numeric(row["response"])
        lookup_y[q_idx][row["response"]] = y
    except Exception as _:
        # Get all rows with same question ID
        curr_q = df[df["question_id"] == q_idx]
        # Create dict entry with list of unique responses
        lookup_y[q_idx] = {v: k for k, v in enumerate(curr_q["response"].unique())}
        y = lookup_y[q_idx][row["response"]]
    finally:
        df.loc[idx, "y"] = y

In [15]:
df.head()

,question_id,size_treatment,occluded_concepts,response,y
0,337814001,1,[28-table],3,3.0
1,337814001,1,[6-wall],3,3.0
2,337814001,1,[26-chair],3,3.0
3,337814001,1,[2-leg],3,3.0
4,337814001,1,[1-wall],3,3.0


In [16]:
# Make dictionary of dataframes w.r.t. question id
df_dict = {q_idx: df[df["question_id"] == q_idx] for q_idx in dict_counter_resp}

In [25]:
df_dict[399744001].head()

,question_id,size_treatment,occluded_concepts,response,y
111321,399744001,1,[33-man],13,13.0
111322,399744001,1,[7-glass],13,13.0
111323,399744001,1,[5-face],13,13.0
111324,399744001,1,[1-head],13,13.0
111325,399744001,1,[30-man],13,13.0


Let us make binary columns indicating whether a concept was occluded or not.
By default, all concepts are present (1). Then, we zero out the ones occluded.

In [29]:
# Adding 1 column (of zeros) to each sub-dataframe for each concept found
for q_idx in df_dict:
    for concept in dict_concepts[q_idx]:
        df_dict[q_idx][concept] = 1

In [30]:
df_dict[399744001].head()

,question_id,size_treatment,occluded_concepts,response,y,33-man,7-glass,5-face,1-head,30-man,10-hair,31-hat,16-head,11-sunglass,3-face,26-shirt,13-hair,20-head,9-ceiling,6-head
111321,399744001,1,[33-man],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111322,399744001,1,[7-glass],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111323,399744001,1,[5-face],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111324,399744001,1,[1-head],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111325,399744001,1,[30-man],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [ ]:
# Correcting the dataframes based on occlusions
for q_idx in df_dict:
    for id, row in df_dict[q_idx].iterrows():
        curr_concepts = row["concepts"]
        for c in curr_concepts:
            df_dict[q_idx].loc[id, c] = 0

    # TODO: continue impl. from here

```df_dict```
- Dictionary with question_ids as keys for which we have valid RKs
- Each entry in the dictionary is a dataframe with
  - The original response (no occlusion; size_treatment=0)
  - A bunch of rows with the counterfactual responses
  - A series of columns matching 1:1 the concepts found

---

Loading pickled graphs to get concept lists

In [6]:
graphs_folder = f"./rk/{MODEL}/{DATASET}/graphs_alt"
graphs_list = os.listdir(graphs_folder)
graphs_dict = {}

for q_idx in counter_question_ids:
    file_name = f"{graphs_folder}/graph_alt_{q_idx}.pkl"
    graphs_dict[q_idx] = data_io.load_graph_pickle(file_name)

In [16]:
# For each concept column, we now set its values depending on the list of concepts in 'concepts'
for q_idx in df_dict:
    for id, row in df_dict[q_idx].iterrows():
        curr_concepts = row["concepts"]
        for c in curr_concepts:
            df_dict[q_idx].loc[id, c] = 1

    # Adding 1-hot-like rows to current dataframe
    new_row = df_dict[q_idx].iloc[-1].to_dict()
    new_row["size"] = ["0"]
    keys_to_ignore = ["question_id", "response", "size", "y"]
    for concepts in dict_concepts[q_idx]:
        for k in new_row:
            if k in keys_to_ignore:
                continue
            new_row[k] = [0]
        # Concat new line
        df_dict[q_idx] = pd.concat(
            [df_dict[q_idx], pd.DataFrame.from_dict(new_row)], ignore_index=True
        )

    # Cleaning up the dict: the 'concepts' column is not useful anymore
    df_dict[q_idx].drop("concepts", axis=1, inplace=True)

Up to this point, we have the following
- A dictionary with as many keys as questions for which we have valid RKs
- Each entry in the dictionary is a dataframe with
  - The original response + total num of concepts (under 'size') + all 1s indicating that all the concepts are present
  - A bunch of rows corresponding to the counterfactual responses + the size of the 'constraining' RK + 1s and 0s depending on which concepts are present

### Running causal analysis

In [ ]:
def causal_inference(data, dot_graph, nx_graph, return_df=False, run_refutation=False):
    res_ci = []
    all_estimates = []
    refutations = []
    for col in nx_graph.nodes:
        if col == "y":
            continue
        print(f"Working on {col}...")

        model = CausalModel(
            data=data,
            treatment=col,
            outcome="y",
            graph=dot_graph,
            missing_nodes_as_confounders=True,
        )

        estimand = model.identify_effect(proceed_when_unidentifiable=True)

        ce_estimate = model.estimate_effect(
            estimand,
            method_name="backdoor.generalized_linear_model",
            control_value=0,
            treatment_value=1,
            test_significance=True,
            fit_estimator=True,
            method_params={"glm_family": sm.families.Binomial()},
        )

        # Run refutation with Placebo
        if run_refutation:
            res_refutation = model.refute_estimate(
                estimand,
                ce_estimate,
                method_name="placebo_treatment_refuter",
                show_progress_bar=False,
                placebo_type="permute",
            )

            refutations.append(
                {
                    col: res_refutation,
                }
            )

        all_estimates.append(ce_estimate)
        res_ci.append({"treatment": col, "ce": str(ce_estimate.value)})

    sorted_res_ci = sorted(res_ci, key=lambda x: x["ce"], reverse=True)

    if return_df:
        return pd.DataFrame.from_dict(sorted_res_ci), all_estimates, refutations
    else:
        return sorted_res_ci, all_estimates, refutations

For each graph, add node representing model responses

In [ ]:
# Isolate the original graphs
og_graphs_dict = {k: graphs_dict[k][dict_max_triplets[k]][0] for k in graphs_dict}

# Add 'response' node
for og_g in og_graphs_dict.values():
    node_list = list(og_g.nodes())
    outcome_node = "y"
    og_g.add_node(outcome_node)

    for n in node_list:
        og_g.add_edge(n, outcome_node)

In [19]:
# Ad-hoc fixes to comply with pydot formatting
if DATASET == "seed" and MODEL == "LLaVa-1.6":
    # Fix column name 'flagpole, material: wood' for question with ID 4106
    idx_to_fix = "4106"
    df_dict[idx_to_fix].rename(
        {"flagpole, material: wood": "wood"}, axis=1, inplace=True
    )
    # Do the same for the graph file
    og_graphs_dict[idx_to_fix] = nx.relabel_nodes(
        og_graphs_dict[idx_to_fix], {"flagpole, material: wood": "wood"}
    )
    og_graphs_dict[idx_to_fix].nodes["wood"]["label"] = "wood"

    # Fix edge property for question with ID 2740
    idx_to_fix = "2740"
    og_graphs_dict[idx_to_fix]["groceries"]["bottles"]["label"] = "contains"

In [ ]:
res = {}
estimates = {}
refutations = {}

for g_idx, graph in og_graphs_dict.items():
    # make out dir
    curr_out_dir = f"{CE_OUT_DIR}/{MODEL}/{DATASET}"
    data_io.make_dir(curr_out_dir)

    print(f">> Working with {g_idx}")
    print(graph.nodes)
    # removing self loops for causal analysis
    graph.remove_edges_from(nx.selfloop_edges(graph))
    # create temporary file for causal inference
    temp_file_path = "./temp.dot"
    nx.drawing.nx_pydot.write_dot(graph, temp_file_path)
    save_graph_to_img(g_idx, graph, curr_out_dir)
    res[g_idx], estimates[g_idx], refutations[g_idx] = causal_inference(
        df_dict[g_idx], temp_file_path, graph, run_refutation=False
    )

Folder ./causal_effects/ShareGPT4V/seed already exists.
>> Working with 3981
['people', 'tree', 'y']
Working on people...
Working on tree...
Folder ./causal_effects/ShareGPT4V/seed already exists.
>> Working with 1656
['car', 'image', 'y']
Folder ./causal_effects/ShareGPT4V/seed/graphs_img already exists.
Working on car...
Working on image...
Folder ./causal_effects/ShareGPT4V/seed already exists.
>> Working with 1620
['musicians', 'musical instruments', 'y']
Folder ./causal_effects/ShareGPT4V/seed/graphs_img already exists.
Working on musicians...
Working on musical instruments...
Folder ./causal_effects/ShareGPT4V/seed already exists.
>> Working with 2342
['stage', 'lights', 'y']
Folder ./causal_effects/ShareGPT4V/seed/graphs_img already exists.
Working on stage...
Working on lights...
Folder ./causal_effects/ShareGPT4V/seed already exists.
>> Working with 1948
['liquor bottles', 'shelves', 'y']
Folder ./causal_effects/ShareGPT4V/seed/graphs_img already exists.
Working on liquor bott

In [22]:
# Write code to save results to file
for q_idx in res:
    curr_df = pd.DataFrame.from_dict(res[q_idx])
    curr_df.to_csv(f"{curr_out_dir}/ce_{q_idx}.csv")

In [ ]:
# Saving refutation results
out_text = ""
for r in refutations:
    out_text += f">> Refutations results for {r}.\n\n"
    for v in refutations[r]:
        for treatment in v:
            out_text += f'~ Current treatment variable: "{treatment}"\n'
            out_text += f"{v[treatment]}\n"
    out_text += "=" * 30
    out_text += "\n"

with open(f"{curr_out_dir}/refutations_{DATASET}.txt", "w") as fp:
    fp.write(out_text)